# BNPL — Merchant 3-Month GMV Forecast (13 Weeks)

**Goal:** Forecast each merchant’s next 13 weeks of **gross merchandise value (GMV)** from historical transactions, validate on a held-out future 13-week period, and export `predicted_3m_revenue` for merchant ranking.

**Method:** Global Histogram Gradient Boosting regression on merchant-week historical features. Training labels are the *following* 13 weeks of GMV. Compare with a trailing-13-week baseline. **No random train/test split and no future-derived predictors.**

**Terminology:** `predicted_3m_revenue` is **predicted merchant transaction value (GMV)**, *not* BNPL fee revenue or profit. This forecast assumes the transaction history is representative of the prospective BNPL business; explain that limitation in your presentation.

**Inputs:** `data/curated/curated_transactions/part_2.parquet`, `part_3.parquet`, `part_4.parquet`. **Output:** `data/curated/merchant_features/predicted_3m_revenue.parquet` (separate file; does not overwrite merchant features).

## 1. Dependencies and paths
If necessary, install dependencies in your active notebook environment: `%pip install duckdb scikit-learn pyarrow`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import duckdb

ROOT = Path.cwd()
if not (ROOT / 'data' / 'curated' / 'curated_transactions').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'curated' / 'curated_transactions'
OUT_DIR = ROOT / 'data' / 'curated' / 'merchant_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = [DATA_DIR / f'part_{i}.parquet' for i in (2, 3, 4)]
assert all(p.exists() for p in FILES), f'Missing parquet files: {[str(p) for p in FILES if not p.exists()]}'
HORIZON = 13
SEED = 42
print('Root:', ROOT)
print('Files:', FILES)

## 2. Aggregate transactions into weekly merchant GMV

DuckDB reads the Parquet files without loading all raw transactions into pandas. We deduplicate identical `order_id` records using one record per order; conflicting records with the same order ID should be resolved upstream in ETL. Weeks start on Monday. The final incomplete calendar week is excluded to avoid treating a partial week as a full observation.

In [ ]:
con = duckdb.connect()
paths = '[' + ', '.join(repr(str(p)) for p in FILES) + ']'
# One row per order_id; ETL should have already resolved conflicting duplicates.
query = f"""
WITH orders AS (
    SELECT order_id,
           ANY_VALUE(merchant_abn) AS merchant_abn,
           ANY_VALUE(CAST(order_datetime AS TIMESTAMP)) AS order_datetime,
           ANY_VALUE(TRY_CAST(dollar_value AS DOUBLE)) AS dollar_value,
           ANY_VALUE(consumer_id) AS consumer_id
    FROM read_parquet({paths}, union_by_name=true)
    WHERE order_id IS NOT NULL AND merchant_abn IS NOT NULL
    GROUP BY order_id
), valid AS (
    SELECT * FROM orders
    WHERE order_datetime IS NOT NULL AND dollar_value IS NOT NULL
      AND dollar_value >= 0
)
SELECT merchant_abn,
       CAST(date_trunc('week', order_datetime) AS DATE) AS week,
       SUM(dollar_value) AS gmv,
       COUNT(*) AS txn_count,
       COUNT(DISTINCT consumer_id) AS unique_users
FROM valid
GROUP BY merchant_abn, week
ORDER BY merchant_abn, week
"""
weekly = con.execute(query).df()
con.close()
weekly['week'] = pd.to_datetime(weekly['week'])
assert not weekly.empty, 'No weekly data found'
# The latest transaction week may be incomplete; drop it conservatively.
last_week = weekly['week'].max()
weekly = weekly.loc[weekly['week'] < last_week].copy()
assert not weekly.empty
print('Complete weeks through:', weekly['week'].max().date())
print('Merchants:', weekly.merchant_abn.nunique(), 'merchant-weeks with activity:', len(weekly))
print(weekly.head())

## 3. Construct a complete weekly panel

Missing merchant-weeks are represented as **zero sales**. This is important: a missing row does not mean that a merchant had unknown sales. It means no recorded orders in that week. New merchants have no artificial history before their first recorded transaction; tenure is measured from first observed activity.

In [ ]:
all_merchants = np.sort(weekly['merchant_abn'].unique())
all_weeks = pd.date_range(weekly['week'].min(), weekly['week'].max(), freq='W-MON')
index = pd.MultiIndex.from_product([all_merchants, all_weeks], names=['merchant_abn','week'])
panel = (weekly.set_index(['merchant_abn','week'])[['gmv','txn_count','unique_users']]
         .reindex(index, fill_value=0).reset_index())
for c in ['gmv','txn_count','unique_users']:
    panel[c] = panel[c].astype('float32')
print('Panel rows:', len(panel), '| weeks:', len(all_weeks), '| merchants:', len(all_merchants))
assert len(all_weeks) >= 45, 'Need >=45 complete weeks for historical windows, training and validation.' 

## 4. Engineer strictly historical predictors and future labels

At prediction week **t**, predictors use observations through t only. The label is GMV in weeks **t+1 through t+13**. No target-derived feature uses any part of the forecast window.

In [ ]:
g = panel.groupby('merchant_abn', sort=False)
for window in (4, 8, 13, 26):
    panel[f'gmv_sum_{window}'] = g['gmv'].transform(lambda x: x.rolling(window, min_periods=1).sum()).astype('float32')
for window in (4, 13):
    panel[f'txn_sum_{window}'] = g['txn_count'].transform(lambda x: x.rolling(window, min_periods=1).sum()).astype('float32')
    panel[f'users_mean_{window}'] = g['unique_users'].transform(lambda x: x.rolling(window, min_periods=1).mean()).astype('float32')
panel['gmv_mean_13'] = panel['gmv_sum_13'] / 13.0
panel['gmv_std_13'] = g['gmv'].transform(lambda x: x.rolling(13, min_periods=2).std()).fillna(0).astype('float32')
panel['gmv_lag_1'] = g['gmv'].shift(1).fillna(0).astype('float32')
panel['gmv_lag_13'] = g['gmv'].shift(13).fillna(0).astype('float32')
panel['gmv_trend_4_vs_13'] = (panel['gmv_sum_4'] / 4) / (panel['gmv_sum_13'] / 13 + 1) - 1
panel['active_weeks_13'] = g['gmv'].transform(lambda x: x.gt(0).rolling(13, min_periods=1).sum()).astype('float32')
panel['weeks_observed'] = g.cumcount().astype('float32') + 1
panel['week_of_year'] = panel['week'].dt.isocalendar().week.astype(int)
panel['season_sin'] = np.sin(2*np.pi*panel['week_of_year']/52.18)
panel['season_cos'] = np.cos(2*np.pi*panel['week_of_year']/52.18)
# A direct 13-week future target. All 13 weeks must be present.
future_parts = [g['gmv'].shift(-k) for k in range(1, HORIZON+1)]
panel['target_13w_gmv'] = sum(future_parts)
FEATURES = ['gmv_sum_4','gmv_sum_8','gmv_sum_13','gmv_sum_26',
            'txn_sum_4','txn_sum_13','users_mean_4','users_mean_13',
            'gmv_mean_13','gmv_std_13','gmv_lag_1','gmv_lag_13',
            'gmv_trend_4_vs_13','active_weeks_13','weeks_observed',
            'season_sin','season_cos']
assert panel.loc[panel.week == all_weeks[-1], 'target_13w_gmv'].isna().all()
print('Features:', FEATURES)
print('Target missing (expected for final 13 weeks):', panel.target_13w_gmv.isna().mean().round(3))

## 5. Honest time-based backtest

Validation cutoff is **13 weeks before the last complete week**. Training examples are cutoffs **at least another 13 weeks earlier**, so their entire outcome window is known by the validation cutoff. We sample training cutoffs every four weeks to reduce overlapping labels and training time. This is a *single holdout*, not proof of generalisation across multiple market regimes.

In [ ]:
last_complete_week = all_weeks[-1]
validation_cutoff = all_weeks[-(HORIZON + 1)]
# Require 13 observed weeks before each training cutoff; last training outcome ends by validation cutoff.
training_cutoffs = all_weeks[12 : len(all_weeks)-2*HORIZON : 4]
assert len(training_cutoffs) >= 2, 'Insufficient history for leakage-free 13-week backtest.'
train = panel.loc[panel.week.isin(training_cutoffs) & panel.target_13w_gmv.notna()].copy()
valid = panel.loc[panel.week == validation_cutoff].copy()
assert len(valid) == len(all_merchants)
assert train.week.max() + pd.Timedelta(weeks=HORIZON) <= validation_cutoff
assert valid.target_13w_gmv.notna().all()
print('Train cutoffs:', [d.strftime('%Y-%m-%d') for d in training_cutoffs])
print('Last training target ends:', (train.week.max() + pd.Timedelta(weeks=HORIZON)).date())
print('Validation forecast origin:', validation_cutoff.date())
print('Validation target through:', last_complete_week.date())
print('Training rows:', len(train), 'Validation merchants:', len(valid))

## 6. Fit the model and compare against a simple baseline

In [ ]:
def make_model():
    return HistGradientBoostingRegressor(
        loss='squared_error', max_iter=240, learning_rate=0.05,
        max_leaf_nodes=25, min_samples_leaf=40, l2_regularization=2.0,
        early_stopping=True, validation_fraction=0.12,
        random_state=SEED
    )

# Model the log of 13-week GMV to reduce extreme-value dominance.
model = make_model()
X_train = train[FEATURES].replace([np.inf,-np.inf], np.nan)
y_train = np.log1p(train['target_13w_gmv'].clip(lower=0))
model.fit(X_train, y_train)
X_valid = valid[FEATURES].replace([np.inf,-np.inf], np.nan)
valid['model_pred'] = np.expm1(model.predict(X_valid)).clip(min=0)
valid['baseline_pred'] = valid['gmv_sum_13'].clip(lower=0)

def metrics(y, pred):
    y = np.asarray(y, dtype=float); pred = np.asarray(pred, dtype=float)
    return {'MAE':mean_absolute_error(y,pred),
            'RMSE':np.sqrt(mean_squared_error(y,pred)),
            'WAPE':np.abs(y-pred).sum() / max(y.sum(),1)*100,
            'Total predicted GMV':pred.sum(), 'Total actual GMV':y.sum()}
comparison = pd.DataFrame({
    'Historical 13-week baseline':metrics(valid.target_13w_gmv, valid.baseline_pred),
    'Histogram gradient boosting':metrics(valid.target_13w_gmv, valid.model_pred)
}).T
print(comparison.round(2).to_string())
comparison.to_csv(OUT_DIR / 'forecast_validation_metrics.csv')

## 7. Inspect prediction quality

Look at error distributions, outliers, and whether the new model **actually beats** the baseline. If it does not, do not claim it is more accurate: investigate or use the baseline until you have evidence of improvement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(np.log1p(valid.target_13w_gmv), np.log1p(valid.model_pred), s=5, alpha=.25)
axes[0].plot([0,16],[0,16], ls='--', color='black')
axes[0].set(xlabel='log1p(actual next 13-week GMV)', ylabel='log1p(predicted)', title='Held-out forecasts vs actuals')
axes[1].hist(np.log1p(valid.target_13w_gmv), bins=50, alpha=.6, label='Actual')
axes[1].hist(np.log1p(valid.model_pred), bins=50, alpha=.6, label='Predicted')
axes[1].set(xlabel='log1p(13-week GMV)', ylabel='Merchants', title='Forecast distribution')
axes[1].legend(); plt.tight_layout(); plt.show()
valid[['merchant_abn','week','target_13w_gmv','model_pred','baseline_pred']].to_csv(
    OUT_DIR / 'forecast_backtest_predictions.csv', index=False)

## 8. Refit using all known history, then forecast the next 13 weeks

Once validation is complete, train on additional labelled cutoffs whose 13-week outcomes end by the latest complete week. Predict at the **latest complete week**, for which the next 13 weeks are not in the dataset. The training sample excludes the prediction cutoff itself.

In [ ]:
final_cutoffs = all_weeks[12 : len(all_weeks)-HORIZON : 4]
# Include the most recent fully-labelled cutoff, even if not on the 4-week grid.
latest_labelled_cutoff = all_weeks[-(HORIZON+1)]
final_cutoffs = sorted(set(final_cutoffs).union({latest_labelled_cutoff}))
final_train = panel.loc[panel.week.isin(final_cutoffs) & panel.target_13w_gmv.notna()].copy()
assert final_train.week.max() + pd.Timedelta(weeks=HORIZON) <= last_complete_week
final_model = make_model()
final_model.fit(final_train[FEATURES].replace([np.inf,-np.inf], np.nan),
                np.log1p(final_train.target_13w_gmv.clip(lower=0)))
latest = panel.loc[panel.week == last_complete_week].copy()
latest['predicted_3m_revenue'] = np.expm1(
    final_model.predict(latest[FEATURES].replace([np.inf,-np.inf], np.nan))
).clip(min=0)
latest['baseline_3m_gmv'] = latest['gmv_sum_13'].clip(lower=0)
print('Forecast origin:', last_complete_week.date())
print('Forecast covers the NEXT 13 weeks (not observed yet).')
print(latest[['merchant_abn','predicted_3m_revenue','baseline_3m_gmv']].head())

## 9. Export merchant predictions and validate coverage

Export one row per merchant. If your ranking dataset contains merchants not found in the weekly transaction panel, include them with a missing prediction and review the mismatch; **do not silently replace missing forecasts with zero**.

In [ ]:
predictions = latest[['merchant_abn','predicted_3m_revenue','baseline_3m_gmv']].copy()
predictions['forecast_origin_week'] = last_complete_week
features_path = OUT_DIR / 'merchant_features.parquet'
if features_path.exists():
    ids = pd.read_parquet(features_path, columns=['merchant_abn'])
    assert ids.merchant_abn.is_unique
    predictions = ids.merge(predictions, on='merchant_abn', how='left', validate='one_to_one')
    print('Merchant features rows:',len(ids),'| unmatched forecasts:',predictions.predicted_3m_revenue.isna().sum())
assert predictions.merchant_abn.is_unique
assert predictions.predicted_3m_revenue.dropna().ge(0).all()
output = OUT_DIR / 'predicted_3m_revenue.parquet'
predictions.to_parquet(output, index=False)
predictions.to_csv(OUT_DIR / 'predicted_3m_revenue.csv', index=False)
print('Saved:', output)
print(predictions.predicted_3m_revenue.describe(percentiles=[.5,.9,.99]).round(2))

## 10. Integrate into ranking (run separately, once)

In `merchant_ranking.ipynb`, load the original merchant features and merge the **new forecast** before scoring. Avoid overwriting `merchant_features.parquet` repeatedly: it can create duplicated/suffixed prediction columns.

```python
features = pd.read_parquet(ROOT / 'data/curated/merchant_features/merchant_features.parquet')
pred = pd.read_parquet(ROOT / 'data/curated/merchant_features/predicted_3m_revenue.parquet')
features = features.drop(columns=['predicted_3m_revenue'], errors='ignore')
merchants = features.merge(pred[['merchant_abn','predicted_3m_revenue']],
                           on='merchant_abn', how='left', validate='one_to_one')
assert merchants.predicted_3m_revenue.notna().all(), 'Investigate missing predictions'
```

Then rerun ranking, sensitivity analysis, and segmentation. **Never merge new predictions into an already-ranked table without recalculating its scores.**

### Presentation limitations
- Forecast horizon = 13 weeks, an approximation to three calendar months.
- The model predicts **merchant GMV**, not BNPL fees, actual onboarding uplift, or profit.
- A single chronological backtest is a first check; additional rolling-origin tests are recommended.
- Feature correlations and seasonal shifts can affect stability. Verify improvement against the baseline before claiming superiority.
- If the historical dataset contains periods after the intended business decision date, explicitly state that the model is a retrospective forecasting exercise rather than a historically available prediction.